# Ensemble Pipeline


In [ ]:
def run_ensemble(fused_path: Path, ir_path: Path, vis_path: Path, yolo_model_obj,
                 conf: float = cascade_conf) -> dict:
    """Run YOLO on the fused image, then refine boxes with SAM 2 on visible RGB."""
    fused_bgr = cv2.imread(str(fused_path))
    if fused_bgr is None:
        raise FileNotFoundError(f"Fused image not found: {fused_path}")
    fused_rgb = cv2.cvtColor(fused_bgr, cv2.COLOR_BGR2RGB)

    ir_gray = cv2.imread(str(ir_path), cv2.IMREAD_GRAYSCALE)
    visible_rgb = resize_visible_for_sam(vis_path, TARGET_RES)

    preds = yolo_model_obj.predict(source=str(fused_path), conf=conf, iou=nms_iou,
                                   imgsz=TARGET_RES, max_det=max_det, verbose=False)
    boxes = preds[0].boxes.xyxy.cpu().numpy()
    confs = preds[0].boxes.conf.cpu().numpy()

    masks, scores, mask_boxes, veto_reasons = run_sam2_refinement(visible_rgb, boxes)

    return {
        "image_name": fused_path.name,
        "fused_image": fused_rgb,
        "infrared_image": ir_gray,
        "visible_image": visible_rgb,
        "yolo_boxes_xyxy": boxes,
        "yolo_conf": confs,
        "final_masks": masks,
        "final_scores": scores,
        "final_mask_boxes_xyxy": mask_boxes,
        "vetoed": len(veto_reasons),
        "veto_reasons": veto_reasons,
    }


In [ ]:
val_fused_paths = sorted((processed_dataset_dir / "val" / "images").glob("*.jpg"))
val_ir_paths = [llvip_infrared_train_dir / p.name for p in val_fused_paths]
val_vis_paths = [llvip_visible_train_dir / p.name for p in val_fused_paths]

print(f"Validation images for ensemble: {len(val_fused_paths)}")


Validation images for ensemble: 1804


In [ ]:
ensemble_yolo = YOLO(str(best_checkpoint_path))
all_results = []

for i, (fp, ip, vp) in enumerate(zip(val_fused_paths, val_ir_paths, val_vis_paths)):
    result = run_ensemble(fp, ip, vp, ensemble_yolo, conf=cascade_conf)
    all_results.append(result)
    if (i + 1) % 100 == 0:
        print(f"{i+1}/{len(val_fused_paths)} images processed")

print(f"Ensemble finished on {len(all_results)} images.")


100/1804 images processed
200/1804 images processed
300/1804 images processed
400/1804 images processed
500/1804 images processed
600/1804 images processed
700/1804 images processed
800/1804 images processed
900/1804 images processed
1000/1804 images processed
1100/1804 images processed
1200/1804 images processed
1300/1804 images processed
1400/1804 images processed
1500/1804 images processed
1600/1804 images processed
1700/1804 images processed
1800/1804 images processed
Ensemble finished on 1804 images.


In [ ]:
# Anoether heavy cell. Save in case runtime disconnects.

import json
import numpy as np
from pathlib import Path

# Ensure directory exists
ensemble_results_dir.mkdir(parents=True, exist_ok=True)
ensemble_results_cache_path = ensemble_results_dir / f"ensemble_results_cache_conf_{cascade_conf:.2f}.json"

ensemble_results_cache = {
    "metadata": {
        "cascade_conf": float(cascade_conf),
        "min_area_ratio": float(SAM_MASK_MIN_AREA_RATIO),
        "max_area_ratio": float(SAM_MASK_MAX_AREA_RATIO),
        "min_box_iou": float(SAM_MASK_MIN_BOX_IOU),
        "num_images": len(all_results),
        "image_size": int(TARGET_RES),
        "checkpoint_path": str(best_checkpoint_path),
    },
    "results": []
}

for res in all_results:
    # use .get() with empty lists as fallbacks to prevent the KeyError
    # if a specific image had zero detections.
    ensemble_results_cache["results"].append({
        "image_name": res["image_name"],
        "yolo_boxes_xyxy": np.asarray(res.get("yolo_boxes_xyxy", [])).tolist(),
        "yolo_conf": np.asarray(res.get("yolo_conf", [])).tolist(),
        "final_boxes_xyxy": np.asarray(res.get("final_boxes_xyxy", [])).tolist(),
        "final_mask_boxes_xyxy": np.asarray(res.get("final_mask_boxes_xyxy", [])).tolist(),
        "final_scores": np.asarray(res.get("final_scores", [])).tolist(),
        "vetoed": int(res.get("vetoed", 0)),
        "veto_reasons": res.get("veto_reasons", []),
    })

with open(ensemble_results_cache_path, "w") as f:
    json.dump(ensemble_results_cache, f, indent=2)

print(f"Successfully saved ensemble results: {ensemble_results_cache_path}")

Successfully saved ensemble results: /home/vteam5/multispectral_pedestrian_ensemble/results_ensemble_v2/ensemble_results_cache_conf_0.30.json


In [ ]:
# run only when runtime disconnects

import json
import numpy as np
import os

ensemble_results_cache_path = ensemble_results_dir / f"ensemble_results_cache_conf_{cascade_conf:.2f}.json"

if os.path.exists(ensemble_results_cache_path):
    print(f"🔄 Loading cache from {ensemble_results_cache_path}...")

    with open(ensemble_results_cache_path, "r") as f:
        cache_data = json.load(f)

    all_results = []
    for item in cache_data["results"]:
        all_results.append({
            "image_name": item["image_name"],
            "yolo_boxes_xyxy": np.array(item["yolo_boxes_xyxy"]),
            "yolo_conf": np.array(item["yolo_conf"]),
            "final_boxes_xyxy": np.array(item["final_boxes_xyxy"]),
            "final_mask_boxes_xyxy": np.array(item["final_mask_boxes_xyxy"]),
            "final_scores": np.array(item["final_scores"]),
            "vetoed": bool(item["vetoed"]),
            "veto_reasons": item["veto_reasons"]
        })

    # Sync metadata
    cascade_conf = cache_data["metadata"]["cascade_conf"]
    print(f"✅ Session restored. {len(all_results)} images loaded.")
else:
    print(f"⚠️ No cache file found at {ensemble_results_cache_path}.")